In [ ]:
import json
import os
import re
import sys

from tqdm import tqdm
from pathlib import Path
from typing import Optional

import time
from transformers import AutoTokenizer
from vllm import LLM, SamplingParams

In [3]:
# ── Configuration ─────────────────────────────────────────────────────────────
MODEL_ID    = "Qwen/Qwen3-4B-Thinking-2507"
GPU_ID      = "0"                    # CUDA_VISIBLE_DEVICES
DATA_PATH   = "data/public.jsonl"
OUTPUT_PATH = "results/starter_results.jsonl"
MAX_TOKENS = 2048 # 32768

# os.environ["CUDA_VISIBLE_DEVICES"] = GPU_ID

In [4]:
data = [json.loads(line) for line in open(DATA_PATH)]
data = data[:5]

n_mcq  = sum(bool(d.get("options")) for d in data)
n_free = sum(not d.get("options")   for d in data)
print(f"Loaded {len(data)} questions  ({n_mcq} MCQ, {n_free} free-form)")

Loaded 5 questions  (2 MCQ, 3 free-form)


In [5]:
SYSTEM_PROMPT_MATH = (
    "You are an expert mathematician. Solve the problem step-by-step. "
    "Put your final answer inside \\boxed{}. "
    "If the problem has multiple sub-answers, separate them by commas inside a single \\boxed{}, "
    "e.g. \\boxed{3, 7}."
)

SYSTEM_PROMPT_MCQ = (
    "You are an expert mathematician. "
    "Read the problem and the answer choices below, then select the single best answer. "
    "Output ONLY the letter of your chosen option inside \\boxed{}, e.g. \\boxed{C}."
)


def build_prompt(question: str, options: Optional[list]) -> tuple[str, str]:
    """Return (system_prompt, user_prompt) for a question."""
    if options:
        labels    = [chr(65 + i) for i in range(len(options))]
        opts_text = "\n".join(f"{lbl}. {opt.strip()}" for lbl, opt in zip(labels, options))
        return SYSTEM_PROMPT_MCQ, f"{question}\n\nOptions:\n{opts_text}"
    return SYSTEM_PROMPT_MATH, question

In [7]:
MODEL_ID = "Qwen/Qwen3-4B-Thinking-2507"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token

llm = LLM(
    model=MODEL_ID,
    quantization="bitsandbytes",
    load_format="bitsandbytes",
    enable_prefix_caching=False,
    gpu_memory_utilization=0.50,
    max_model_len=16384,
    trust_remote_code=True,
    max_num_seqs=256,
    max_num_batched_tokens=32768,
)

sampling_params = SamplingParams(
    max_tokens=MAX_TOKENS,
    temperature=0.6,
    top_p=0.95,
    top_k=20,
    min_p=0.0,
    presence_penalty=0.0,
    repetition_penalty=1.0,
)

print("Model loaded.")

INFO 05-03 10:21:57 [utils.py:233] non-default args: {'trust_remote_code': True, 'load_format': 'bitsandbytes', 'max_model_len': 16384, 'enable_prefix_caching': False, 'gpu_memory_utilization': 0.5, 'max_num_batched_tokens': 32768, 'max_num_seqs': 256, 'disable_log_stats': True, 'quantization': 'bitsandbytes', 'model': 'Qwen/Qwen3-4B-Thinking-2507'}
INFO 05-03 10:21:58 [model.py:555] Resolved architecture: Qwen3ForCausalLM
INFO 05-03 10:21:58 [model.py:1680] Using max model len 16384
INFO 05-03 10:21:58 [scheduler.py:239] Chunked prefill is enabled with max_num_batched_tokens=32768.
INFO 05-03 10:22:00 [kernel.py:205] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['native'])
(EngineCore pid=4556) INFO 05-03 10:22:01 [core.py:109] Initializing a V1 LLM engine (v0.20.1) with config: model='Qwen/Qwen3-4B-Thinking-2507', speculative_config=None, tokenizer='Qwen/Qwen3-4B-Thinking-2507', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, tokeni

Loading safetensors checkpoint shards:   0% Completed | 0/3 [00:00<?, ?it/s]
(EngineCore pid=4556) /workspace/.venv/lib/python3.12/site-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
(EngineCore pid=4556)   torch._check_is_size(blocksize)
Loading safetensors checkpoint shards:  33% Completed | 1/3 [00:29<00:58, 29.38s/it]
Loading safetensors checkpoint shards:  67% Completed | 2/3 [01:03<00:32, 32.25s/it]
Loading safetensors checkpoint shards: 100% Completed | 3/3 [01:04<00:00, 17.88s/it]
Loading safetensors checkpoint shards: 100% Completed | 3/3 [01:04<00:00, 21.47s/it]
(EngineCore pid=4556) 


(EngineCore pid=4556) INFO 05-03 10:23:18 [gpu_model_runner.py:4879] Model loading took 2.7 GiB memory and 73.190355 seconds
(EngineCore pid=4556) INFO 05-03 10:23:43 [backends.py:1069] Using cache directory: /root/.cache/vllm/torch_compile_cache/5e32fa7223/rank_0_0/backbone for vLLM's torch.compile
(EngineCore pid=4556) INFO 05-03 10:23:43 [backends.py:1128] Dynamo bytecode transform time: 24.97 s
(EngineCore pid=4556) INFO 05-03 10:23:54 [backends.py:376] Cache the graph of compile range (1, 32768) for later use
(EngineCore pid=4556) INFO 05-03 10:24:04 [backends.py:391] Compiling a graph for compile range (1, 32768) takes 20.15 s
(EngineCore pid=4556) INFO 05-03 10:24:11 [decorators.py:668] saved AOT compiled function to /root/.cache/vllm/torch_compile_cache/torch_aot_compile/19b7976dfd380c01c4c594774f153791da26941be5053e4d99d9c191bfb391f4/rank_0_0/model
(EngineCore pid=4556) INFO 05-03 10:24:11 [monitor.py:53] torch.compile took 52.73 s in total
(EngineCore pid=4556) INFO 05-03 10:

Capturing CUDA graphs (mixed prefill-decode, PIECEWISE):  98%|█████████▊| 50/51 [00:10<00:00,  5.00it/s]/workspace/.venv/lib/python3.12/site-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
(EngineCore pid=4556)   torch._check_is_size(blocksize)
Capturing CUDA graphs (mixed prefill-decode, PIECEWISE): 100%|██████████| 51/51 [00:10<00:00,  4.67it/s]
Capturing CUDA graphs (decode, FULL): 100%|██████████| 35/35 [00:06<00:00,  5.03it/s]


(EngineCore pid=4556) INFO 05-03 10:25:05 [gpu_model_runner.py:6133] Graph capturing finished in 19 secs, took 0.97 GiB
(EngineCore pid=4556) INFO 05-03 10:25:05 [gpu_worker.py:599] CUDA graph pool memory: 0.97 GiB (actual), 0.89 GiB (estimated), difference: 0.08 GiB (8.0%).
(EngineCore pid=4556) INFO 05-03 10:25:05 [core.py:299] init engine (profile, create kv cache, warmup model) took 107.41 s (compilation: 52.73 s)
(EngineCore pid=4556) INFO 05-03 10:25:06 [kernel.py:205] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['native'])
Model loaded.


In [8]:
# Build prompts for first 5 entries
prompts = []
for item in data[:5]:
    system, user = build_prompt(item["question"], item.get("options"))
    prompt_text = tokenizer.apply_chat_template(
        [{"role": "system", "content": system},
         {"role": "user",   "content": user}],
        tokenize=False,
        add_generation_prompt=True,
    )
    prompts.append(prompt_text)

# Generate
print(f"Generating responses for {len(prompts)} questions...")
outputs = llm.generate(prompts, sampling_params=sampling_params)

responses = [out.outputs[0].text.strip() for out in outputs]

# Preview first 3
for i in range(min(3, len(responses))):
    print(f"\n── Response {i} (id={data[i].get('id')}) ──")
    print(responses[i][:400], "..." if len(responses[i]) > 400 else "")

Generating responses for 5 questions...


Processed prompts: 100%|██████████| 5/5 [00:56<00:00, 11.24s/it, est. speed input: 15.07 toks/s, output: 160.56 toks/s]


── Response 0 (id=0) ──
This is a complex or challenging question, and it is difficult to provide a direct and correct answer. I need to think about it.
Well, so I need to find the sum of the first 325 positive even whole numbers. Hmm, let's start by recalling what the first few even whole numbers are to make sure I know what "first" means here. Positive even whole numbers start at 2, right? Like 2, 4, 6, 8, ..., so each ...

── Response 1 (id=1) ──
Okay, let's try to solve this integral: the integral from negative infinity to positive infinity of (a^(3/2)) divided by (s² + a²) ds. Hmm, first, I need to check if this makes sense. Wait, the variable of integration is s, right? So the integral is with respect to s. The integrand is a^(3/2) / (s² + a²). Let me write that down:

∫_{-∞}^{+∞} [a^(3/2) / (s² + a²)] ds

First thought: the integral of ...

── Response 2 (id=2) ──
Okay, let's try to solve this problem step by step. First, part (a) is about the turkey cooling down, so I think th

In [9]:
def extract_letter(text: str) -> str:
    m = re.search(r"\\boxed\{([A-Za-z])\}", text)
    if m:
        return m.group(1).upper()
    matches = re.findall(r"\b([A-Z])\b", text.upper())
    return matches[-1] if matches else ""

def score_mcq(response: str, gold_letter: str) -> bool:
    return extract_letter(response) == gold_letter.strip().upper()

# Load Judger for free-form scoring
sys.path.insert(0, ".")
from judger import Judger
judger = Judger(strict_extract=False)

results = []
for item, response in tqdm(zip(data, responses), total=len(data), desc="Scoring"):
    is_mcq = bool(item.get("options"))
    gold   = item["answer"]

    if is_mcq:
        correct = score_mcq(response, str(gold))
    else:
        gold_list = gold if isinstance(gold, list) else [gold]
        try:
            correct = judger.auto_judge(
                pred=response,
                gold=gold_list,
                options=[[]] * len(gold_list),
            )
        except Exception:
            correct = False

    results.append({
        "id":       item.get("id"),
        "is_mcq":   is_mcq,
        "gold":     gold,
        "response": response,
        "correct":  correct,
    })

print(f"Scoring complete. {len(results)} results.")

Scoring: 100%|██████████| 5/5 [00:00<00:00, 24.94it/s]

Scoring complete. 5 results.


In [10]:
mcq_res  = [r for r in results if r["is_mcq"]]
free_res = [r for r in results if not r["is_mcq"]]

def acc(subset):
    return sum(r["correct"] for r in subset) / len(subset) * 100 if subset else 0.0

print("=" * 50)
print("EVALUATION RESULTS")
print("=" * 50)
print(f"  MCQ        : {sum(r['correct'] for r in mcq_res):4d} / {len(mcq_res):4d}  ({acc(mcq_res):.2f}%)")
print(f"  Free-form  : {sum(r['correct'] for r in free_res):4d} / {len(free_res):4d}  ({acc(free_res):.2f}%)")
print(f"  Overall    : {sum(r['correct'] for r in results):4d} / {len(results):4d}  ({acc(results):.2f}%)")
print("=" * 50)

EVALUATION RESULTS
  MCQ        :    0 /    2  (0.00%)
  Free-form  :    1 /    3  (33.33%)
  Overall    :    1 /    5  (20.00%)


In [ ]:
SAVE_EVAL = True   # Set to False when running on the private test set

out_path = Path(OUTPUT_PATH)
out_path.parent.mkdir(parents=True, exist_ok=True)

with open(out_path, "w") as f:
    for r in results:
        if SAVE_EVAL:
            record = {"id": r["id"], "is_mcq": r["is_mcq"], "gold": r["gold"],
                      "response": r["response"], "correct": r["correct"]}
        else:
            record = {"id": r["id"], "is_mcq": r["is_mcq"], "response": r["response"]}
        f.write(json.dumps(record) + "\n")

print(f"Saved {len(results)} records to {out_path}")